# Poverty and Race in the MORPC 15-County Region

This notebook walks through a realistic analysis scenario using `morpc-census`:

1. **Variable discovery** — find the Census tables for poverty status by race
2. **Fetch** — retrieve data for all 15 MORPC counties
3. **Snapshot** — compare poverty rates by race/ethnicity group, 2023
4. **Time series** — track changes in non-white poverty rates, 2019–2023
5. **Map** — choropleth of non-white poverty change by county

> **ACS 5-year note:** Each vintage year covers a 5-year period (e.g., 2023 covers 2019–2023). Consecutive years overlap by four years and are not statistically independent measurements.

In [ ]:
from morpc.logs import config_logs

config_logs('morpc-poverty-race-demo.log', 'debug')

In [ ]:
from morpc_census import (
    Endpoint, Group, CensusAPI, DimensionTable,
    RACE_TABLE_MAP,
    fetch_geos_from_scope_sumlevel,
)
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

## 1. Variable discovery

> **Network required** — the cells below make live calls to the Census API.

Start with an `Endpoint` and search `endpoint.groups` for tables related to the topic. The ACS 5-year survey has hundreds of groups; filtering by keyword narrows the field quickly.

In [ ]:
ep = Endpoint('acs/acs5', 2023)

# Search group descriptions for 'poverty'
{k: v['description'] for k, v in ep.groups.items() if 'poverty' in v['description'].lower()}

In [ ]:
# B17001 — Poverty Status by Sex by Age — is the standard person-level poverty table.
# Inspect its first few estimate variables to understand the label structure.
g = Group(ep, 'B17001')
print(f"Description : {g.description}")
print(f"Universe    : {g.universe}")
print()
# Show the first six estimate codes (skip annotation codes ending in EA/MA)
estimate_vars = [
    (k, v['label'])
    for k, v in g.variables.items()
    if k.endswith('E') and not k.endswith('EA')
]
dict(estimate_vars[:6])

In [ ]:
# B17001 has racial iteration variants B17001A–B17001I.
# Each variant has the same variable structure as B17001 but for a single race/ethnicity group.
# RACE_TABLE_MAP maps the letter suffix to the race label.
print("RACE_TABLE_MAP:")
for code, label in RACE_TABLE_MAP.items():
    print(f"  B17001{code} — {label}")

print()
# Confirm that all racial iteration groups exist in this endpoint
available = {f'B17001{c}': RACE_TABLE_MAP[c] for c in RACE_TABLE_MAP if f'B17001{c}' in ep.groups}
print(f"{len(available)} of {len(RACE_TABLE_MAP)} racial iteration groups available:")
available

## 2. Fetching data

For the poverty-by-race snapshot we only need two variables from each racial iteration table:
- `B17001X_001E` — total population for whom poverty status is determined
- `B17001X_002E` — population with income below the poverty level

Passing all 18 variables in a single `CensusAPI` call is more efficient than 9 separate group fetches. The package infers the group code from each variable name and fetches the human-readable labels automatically.

In [ ]:
# Full B17001 group — total poverty status for all 15 counties.
# We'll use this for the time-series and as a reference for the snapshot.
b17001 = CensusAPI(ep, 'region15', group='B17001')
b17001.long[['geoidfq', 'name', 'variable_label', 'variable', 'estimate', 'moe']].head(6)

In [ ]:
# Fetch just the total and below-poverty counts for every racial group in one request.
race_vars = []
for code in RACE_TABLE_MAP:
    race_vars += [f'B17001{code}_001E', f'B17001{code}_002E']

race_pov = CensusAPI(ep, 'region15', variables=race_vars)
race_pov.long[['name', 'variable_label', 'variable', 'estimate']].head(10)

## 3. Poverty by race — 2023 snapshot

For each racial group, the poverty rate is the share of people in that group whose income falls below the federal poverty level:

$$\text{poverty rate} = \frac{\text{B17001X\_002 (below poverty)}}{\text{B17001X\_001 (total)}}$$

`B17001H` (White alone, not Hispanic or Latino) is the standard reference group for comparisons across racial and ethnic lines.

In [ ]:
# Pivot long → wide: one row per county, one column per variable
wide = race_pov.long.pivot(
    index=['geoidfq', 'name'],
    columns='variable',
    values='estimate',
)
wide.columns.name = None

# Compute poverty rate (%) for each racial group
rates = pd.DataFrame(index=wide.index)
for code, label in RACE_TABLE_MAP.items():
    total = wide[f'B17001{code}_001']
    below = wide[f'B17001{code}_002']
    rates[label] = (below / total * 100).round(1)

# Drop the GEOIDFQ level — keep only county name in the index
rates = rates.reset_index(level='geoidfq', drop=True)
rates.index.name = 'County'
rates

In [ ]:
# Heatmap-style table: higher poverty rates in darker red
rates.style \
    .format('{:.1f}%') \
    .background_gradient(cmap='YlOrRd', axis=None) \
    .set_caption('Poverty Rate by Race/Ethnicity (%), MORPC Region — 2023 ACS 5-Year')

In [ ]:
# Region-wide poverty rate by race (sum across all 15 counties)
whole_region = race_pov.long.groupby('variable')['estimate'].sum()

region_rates = {}
for code, label in RACE_TABLE_MAP.items():
    total = whole_region.get(f'B17001{code}_001', 0)
    below = whole_region.get(f'B17001{code}_002', 0)
    if total > 0:
        region_rates[label] = round(below / total * 100, 1)

region_series = pd.Series(region_rates).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(9, 5))
region_series.plot(kind='barh', ax=ax, color='steelblue', edgecolor='white')
ax.set_title('Poverty Rate by Race/Ethnicity\nMORPC 15-County Region — 2023 ACS 5-Year')
ax.set_xlabel('Poverty Rate (%)')
ax.xaxis.set_major_formatter(mtick.PercentFormatter())
ax.axvline(0, color='black', linewidth=0.8)
for bar, val in zip(ax.patches, region_series):
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height() / 2,
            f'{val}%', va='center', fontsize=9)
plt.tight_layout()
plt.show()

## 4. Non-white poverty over time (2019–2023)

To track change in poverty among non-white residents, we subtract the White alone, not Hispanic or Latino population (B17001H) from the total (B17001). This captures everyone who is not non-Hispanic white — including Hispanic white residents, who are counted in `B17001A` (White Alone) but excluded from `B17001H`.

$$\text{non-white below poverty} = \text{B17001\_002} - \text{B17001H\_002}$$
$$\text{non-white total} = \text{B17001\_001} - \text{B17001H\_001}$$

Fetching four variables per year keeps requests small.

In [ ]:
ts_vars = ['B17001_001E', 'B17001_002E', 'B17001H_001E', 'B17001H_002E']

long_frames = []
for year in range(2019, 2024):
    ep_yr = Endpoint('acs/acs5', year)
    api = CensusAPI(ep_yr, 'region15', variables=ts_vars)
    long_frames.append(api.long)

long_ts = pd.concat(long_frames, ignore_index=True)
long_ts[['name', 'reference_period', 'variable', 'estimate']].head(8)

In [ ]:
# Pivot to one row per county × year
wide_ts = long_ts.pivot_table(
    index=['geoidfq', 'name', 'reference_period'],
    columns='variable',
    values='estimate',
    aggfunc='first',
).reset_index()
wide_ts.columns.name = None

# Non-white poverty rate
wide_ts['non_white_below'] = wide_ts['B17001_002'] - wide_ts['B17001H_002']
wide_ts['non_white_total'] = wide_ts['B17001_001'] - wide_ts['B17001H_001']
wide_ts['non_white_pov_rate'] = (
    wide_ts['non_white_below'] / wide_ts['non_white_total'] * 100
).round(1)

# Table: counties as columns, years as rows
wide_ts.pivot_table(
    index='reference_period',
    columns='name',
    values='non_white_pov_rate',
    aggfunc='first',
)

In [ ]:
pivot_plot = wide_ts.pivot_table(
    index='reference_period',
    columns='name',
    values='non_white_pov_rate',
    aggfunc='first',
)

fig, ax = plt.subplots(figsize=(12, 6))
pivot_plot.plot(ax=ax, marker='o', linewidth=1.5)
ax.set_title('Non-White Poverty Rate by County\nMORPC Region — 2019–2023 ACS 5-Year Estimates')
ax.set_ylabel('Poverty Rate (%)')
ax.set_xlabel('Vintage Year')
ax.set_xticks(range(2019, 2024))
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
ax.legend(
    title='County',
    labels=[n.replace(', Ohio', '') for n in pivot_plot.columns],
    bbox_to_anchor=(1.01, 1),
    loc='upper left',
    fontsize=8,
)
plt.tight_layout()
plt.show()

## 5. Mapping the change

`fetch_geos_from_scope_sumlevel` returns a GeoDataFrame with county boundaries. Merging it with the computed poverty change gives a ready-to-plot choropleth.

A diverging colormap (red = increase, blue = decrease) makes the direction of change immediately readable.

In [ ]:
# Change in non-white poverty rate: 2023 rate minus 2019 rate (percentage points)
change = wide_ts.pivot_table(
    index=['geoidfq', 'name'],
    columns='reference_period',
    values='non_white_pov_rate',
    aggfunc='first',
)
change.columns = [f'rate_{yr}' for yr in change.columns]
change['change_pp'] = (change['rate_2023'] - change['rate_2019']).round(1)
change = change.reset_index()

# County boundaries for the 15-county MORPC region
geos = fetch_geos_from_scope_sumlevel('region15')

In [ ]:
# Join poverty change to geometries
geos_joined = geos.merge(change, left_on='GEOIDFQ', right_on='geoidfq', how='left')

# Symmetric color scale centered at zero
max_abs = geos_joined['change_pp'].abs().max()

fig, ax = plt.subplots(figsize=(10, 8))
geos_joined.plot(
    column='change_pp',
    ax=ax,
    cmap='RdBu_r',
    vmin=-max_abs,
    vmax=max_abs,
    legend=True,
    legend_kwds={
        'label': 'Change in non-white poverty rate (pp)',
        'orientation': 'horizontal',
        'shrink': 0.6,
        'pad': 0.02,
    },
    edgecolor='white',
    linewidth=0.5,
)

# Label each county
for _, row in geos_joined.iterrows():
    cx = row.geometry.centroid.x
    cy = row.geometry.centroid.y
    label = row['NAME'].replace(', Ohio', '').replace(' County', '')
    val = row['change_pp']
    ax.annotate(
        f"{label}\n{val:+.1f}pp",
        (cx, cy),
        ha='center',
        va='center',
        fontsize=6.5,
        color='black',
    )

ax.set_title(
    'Change in Non-White Poverty Rate\nMORPC 15-County Region — 2019 to 2023 ACS 5-Year',
    fontsize=13,
)
ax.axis('off')
plt.tight_layout()
plt.show()